Use ESOL datset and improve the pretrained model by fine-tuning

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from torch import nn

MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME)

In [ ]:
class ChemBERTaRegressor(nn.Module):
    def __init__(self, encoder, hidden_size):
        super().__init__()
        self.encoder = encoder
        self.regressor = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Use CLS token representation
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        return self.regressor(cls_embedding).squeeze(-1)

In [ ]:
4. Prepare ESOL dataset

In [ ]:
def tokenize(smiles):
    return tokenizer(
        smiles,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

In [ ]:
class ESOLDataset(torch.utils.data.Dataset):
    def __init__(self, smiles, labels):
        self.smiles = smiles
        self.labels = labels

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.smiles[idx],
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.float)
        }

In [ ]:
from torch.utils.data import DataLoader

train_dataset = ESOLDataset(train_smiles, train_labels)
val_dataset = ESOLDataset(val_smiles, val_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [ ]:
hidden_size = encoder.config.hidden_size
model = ChemBERTaRegressor(encoder, hidden_size)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
optimizer = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": 2e-5},
    {"params": model.regressor.parameters(), "lr": 1e-4},
])

In [ ]:
from tqdm import tqdm

def train_epoch(model, loader):
    model.train()
    total_loss = 0

    for batch in tqdm(loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        preds = model(input_ids, attention_mask)
        loss = criterion(preds, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
from sklearn.metrics import mean_squared_error

def evaluate(model, loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            preds = model(input_ids, attention_mask)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    rmse = mean_squared_error(labels_all, preds_all, squared=False)
    return rmse

In [ ]:
for epoch in range(5):
    train_loss = train_epoch(model, train_loader)
    val_rmse = evaluate(model, val_loader)

    print(f"Epoch {epoch}: Train Loss={train_loss:.4f}, Val RMSE={val_rmse:.4f}")